In [ ]:
# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow.keras import layers, models
import tensorflow_hub as hub
from sklearn.metrics import classification_report

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

2.13.1


In [2]:
IMAGE_SHAPE = (224, 224)

In [3]:
train_ds, test_ds = tf.keras.utils.image_dataset_from_directory(
    "animal_data",
    validation_split=0.2,
    subset="both",
    shuffle= True,
    labels = "inferred",
    label_mode='int',
    seed=42,
    image_size=IMAGE_SHAPE,
    batch_size=32
)

Found 1944 files belonging to 14 classes.
Using 1556 files for training.
Using 388 files for validation.


In [5]:
efficientnet = "https://www.kaggle.com/models/google/efficientnet-v2/TensorFlow2/imagenet1k-b0-classification/2"

In [ ]:
feature_extractor_layer = hub.KerasLayer(efficientnet,
                                        trainable=True,
                                        name='feature_extraction_layer',
                                        input_shape=IMAGE_SHAPE+(3,)) 


model = models.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    feature_extractor_layer, 
    tf.keras.layers.BatchNormalization(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(14, activation="softmax")])

In [13]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'])

In [15]:
model.fit(train_ds, epochs=3, steps_per_epoch=len(train_ds))

Epoch 1/3


49/49 [==============================] - 117s 2s/step - loss: 1.0638 - accuracy: 0.7018
Epoch 2/3
49/49 [==============================] - 112s 2s/step - loss: 0.3395 - accuracy: 0.9087
Epoch 3/3
49/49 [==============================] - 116s 2s/step - loss: 0.2134 - accuracy: 0.9312


In [16]:
true_test_labels = []
test_predictions = []
for images, labels in test_ds:
    test_predictions.extend(np.argmax(model.predict(images), axis=1))
    true_test_labels.extend(labels.numpy())

1/1 [==============================] - 1s 824ms/step


In [ ]:
print("BEST MODEL: Using Efficient-Net\n\nClassification Report:\n")
print(classification_report(true_test_labels, test_predictions, target_names=test_ds.class_names))